# Career Compass — Person 3 Model Evaluation

This notebook contains the finalized evaluation workflow for the Career Compass recommendation system.

It:

1. Loads the project data and recommendation modules.
2. Builds and scales the occupation feature matrices.
3. Creates six representative evaluation profiles.
4. Defines and validates profile-specific relevant occupation sets.
5. Runs the Baseline, Weighted Compatibility, and Cosine Similarity recommenders, plus the KNN recommender under each of four distance metrics (euclidean, manhattan, minkowski, cosine).
6. Calculates ranking and diversity metrics.
7. Produces the final model-comparison table.

Upload `ADS-599-Capstone-Project-main.zip` to the Colab session before running the notebook.

## 1. Extract the project repository

In [1]:
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/ADS-599-Capstone-Project-main.zip")
EXTRACT_DIR = Path("/content")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Upload {ZIP_PATH.name} to the Colab session before running this cell."
    )

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

PROJECT_DIR = EXTRACT_DIR / "ADS-599-Capstone-Project-main" / "Person_1_Career_Compass"

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}")

print(f"Repository extracted successfully: {PROJECT_DIR}")

FileNotFoundError: Upload ADS-599-Capstone-Project-main.zip to the Colab session before running this cell.

## 2. Install dependencies and import project modules

In [ ]:
%cd /content/ADS-599-Capstone-Project-main/Person_1_Career_Compass
!pip install -q -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd

from IPython.display import display

from src import (
    baseline_model,
    compatibility,
    cosine_model,
    data_prep,
    feature_matrix,
    knn_model,
    scaling,
    user_profile,
)

print("Imports completed successfully.")

## 3. Load the master occupation dataset

In [ ]:
df = data_prep.load_master_data(data_prep.DATA_PATH)

print("Master dataset shape:", df.shape)
display(df.head())

## 4. Build and scale the feature matrices

In [ ]:
matrices = feature_matrix.build_feature_matrices(df)
scaled = scaling.scale_matrices(matrices)

print("Recommendable occupations:", len(matrices.lookup))
print("Interest matrix shape:", matrices.interest.shape)
print("Skill matrix shape:", matrices.skill.shape)
print("Education matrix shape:", matrices.education.shape)
print("Job-zone matrix shape:", matrices.job_zone.shape)
print("Labor-market matrix shape:", matrices.labor_market.shape)

## 5. Create the six evaluation profiles

In [ ]:
evaluation_profiles = [
    user_profile.UserProfile(
        profile_name="Data and Technology",
        interest_realistic=2,
        interest_investigative=5,
        interest_artistic=2,
        interest_social=2,
        interest_enterprising=3,
        interest_conventional=4,
        education_preference=4,
        job_zone_preference=4,
        salary_importance=4,
        employment_importance=4,
        selected_skills=[
            "Programming",
            "Mathematics",
            "Critical thinking",
            "Complex problem solving",
            "Systems analysis",
            "Systems evaluation",
        ],
    ),
    user_profile.UserProfile(
        profile_name="Healthcare",
        interest_realistic=2,
        interest_investigative=4,
        interest_artistic=1,
        interest_social=5,
        interest_enterprising=2,
        interest_conventional=3,
        education_preference=5,
        job_zone_preference=5,
        salary_importance=3,
        employment_importance=4,
        selected_skills=[
            "Science",
            "Active listening",
            "Critical thinking",
            "Service orientation",
            "Social perceptiveness",
            "Judgment and decision making",
        ],
    ),
    user_profile.UserProfile(
        profile_name="Business",
        interest_realistic=1,
        interest_investigative=3,
        interest_artistic=2,
        interest_social=3,
        interest_enterprising=5,
        interest_conventional=4,
        education_preference=4,
        job_zone_preference=4,
        salary_importance=5,
        employment_importance=4,
        selected_skills=[
            "Financial resource management",
            "Personnel management",
            "Negotiation",
            "Persuasion",
            "Judgment and decision making",
            "Time management",
        ],
    ),
    user_profile.UserProfile(
        profile_name="Skilled Trades",
        interest_realistic=5,
        interest_investigative=3,
        interest_artistic=1,
        interest_social=2,
        interest_enterprising=2,
        interest_conventional=4,
        education_preference=2,
        job_zone_preference=3,
        salary_importance=3,
        employment_importance=5,
        selected_skills=[
            "Operation and control",
            "Operations monitoring",
            "Repairing",
            "Troubleshooting",
            "Quality control analysis",
            "Technology design",
        ],
    ),
    user_profile.UserProfile(
        profile_name="Creative Careers",
        interest_realistic=2,
        interest_investigative=3,
        interest_artistic=5,
        interest_social=3,
        interest_enterprising=3,
        interest_conventional=2,
        education_preference=4,
        job_zone_preference=4,
        salary_importance=2,
        employment_importance=3,
        selected_skills=[
            "Writing",
            "Speaking",
            "Active learning",
            "Critical thinking",
            "Technology design",
            "Persuasion",
        ],
    ),
    user_profile.UserProfile(
        profile_name="Social Service",
        interest_realistic=1,
        interest_investigative=4,
        interest_artistic=2,
        interest_social=5,
        interest_enterprising=2,
        interest_conventional=3,
        education_preference=4,
        job_zone_preference=4,
        salary_importance=2,
        employment_importance=4,
        selected_skills=[
            "Active listening",
            "Service orientation",
            "Social perceptiveness",
            "Speaking",
            "Instructing",
            "Coordination",
        ],
    ),
]

print(f"Created {len(evaluation_profiles)} evaluation profiles.")

In [ ]:
profile_summary = pd.DataFrame(
    [
        {
            "Profile": p.profile_name,
            "Realistic": p.interest_realistic,
            "Investigative": p.interest_investigative,
            "Artistic": p.interest_artistic,
            "Social": p.interest_social,
            "Enterprising": p.interest_enterprising,
            "Conventional": p.interest_conventional,
            "Education": p.education_preference,
            "Job Zone": p.job_zone_preference,
            "Salary Importance": p.salary_importance,
            "Employment Importance": p.employment_importance,
            "Selected Skills": ", ".join(p.selected_skills),
        }
        for p in evaluation_profiles
    ]
)

display(profile_summary)

## 6. Define the relevant occupation sets

Each profile has ten manually selected occupations that serve as the evaluation ground truth. O*NET-SOC codes are used instead of titles to prevent matching errors caused by title formatting.

In [ ]:
relevant_occupations = {
    "Data and Technology": {
        "15-2051.00",  # Data Scientists
        "15-2041.00",  # Statisticians
        "15-2041.01",  # Biostatisticians
        "15-2031.00",  # Operations Research Analysts
        "15-2051.01",  # Business Intelligence Analysts
        "15-1252.00",  # Software Developers
        "15-1243.00",  # Database Architects
        "15-1243.01",  # Data Warehousing Specialists
        "15-1212.00",  # Information Security Analysts
        "15-1251.00",  # Computer Programmers
    },
    "Healthcare": {
        "29-1141.00",  # Registered Nurses
        "29-1171.00",  # Nurse Practitioners
        "29-1071.00",  # Physician Assistants
        "29-1123.00",  # Physical Therapists
        "29-1122.00",  # Occupational Therapists
        "29-1051.00",  # Pharmacists
        "29-1126.00",  # Respiratory Therapists
        "29-2032.00",  # Diagnostic Medical Sonographers
        "29-1215.00",  # Family Medicine Physicians
        "29-2042.00",  # Emergency Medical Technicians
    },
    "Business": {
        "13-2011.00",  # Accountants and Auditors
        "15-2051.01",  # Business Intelligence Analysts
        "11-3031.00",  # Financial Managers
        "13-2054.00",  # Financial Risk Specialists
        "13-2051.00",  # Financial and Investment Analysts
        "13-1111.00",  # Management Analysts
        "13-1161.00",  # Market Research Analysts and Marketing Specialists
        "11-2021.00",  # Marketing Managers
        "11-3121.00",  # Human Resources Managers
        "13-1082.00",  # Project Management Specialists
    },
    "Skilled Trades": {
        "49-3011.00",  # Aircraft Mechanics and Service Technicians
        "49-3023.00",  # Automotive Service Technicians and Mechanics
        "47-2031.00",  # Carpenters
        "47-2111.00",  # Electricians
        "49-9021.00",  # HVAC Mechanics and Installers
        "49-9041.00",  # Industrial Machinery Mechanics
        "51-4041.00",  # Machinists
        "47-2152.00",  # Plumbers, Pipefitters, and Steamfitters
        "51-4121.00",  # Welders, Cutters, Solderers, and Brazers
        "47-4021.00",  # Elevator and Escalator Installers and Repairers
    },
    "Creative Careers": {
        "27-1021.00",  # Commercial and Industrial Designers
        "27-1012.00",  # Craft Artists
        "27-1024.00",  # Graphic Designers
        "27-1025.00",  # Interior Designers
        "27-4021.00",  # Photographers
        "27-3043.05",  # Poets, Lyricists and Creative Writers
        "27-1027.00",  # Set and Exhibit Designers
        "27-1014.00",  # Special Effects Artists and Animators
        "15-1255.01",  # Video Game Designers
        "15-1255.00",  # Web and Digital Interface Designers
    },
    "Social Service": {
        "21-1012.00",  # Educational, Guidance, and Career Counselors
        "21-1013.00",  # Marriage and Family Therapists
        "21-1015.00",  # Rehabilitation Counselors
        "21-1019.00",  # Counselors, All Other
        "21-1021.00",  # Child, Family, and School Social Workers
        "21-1022.00",  # Healthcare Social Workers
        "21-1023.00",  # Mental Health and Substance Abuse Social Workers
        "21-1093.00",  # Social and Human Service Assistants
        "21-1094.00",  # Community Health Workers
        "21-1099.00",  # Community and Social Service Specialists, All Other
    },
}

## 7. Validate and display the ground-truth occupations

In [ ]:
catalog_codes = set(matrices.lookup["onet_soc_code"])

validation_rows = [
    {
        "Profile": profile_name,
        "O*NET SOC Code": code,
        "Available": code in catalog_codes,
    }
    for profile_name, relevant_codes in relevant_occupations.items()
    for code in relevant_codes
]

validation_df = pd.DataFrame(validation_rows)

unavailable_df = validation_df.loc[
    ~validation_df["Available"],
    ["Profile", "O*NET SOC Code"],
]

display(validation_df)

if unavailable_df.empty:
    print("All ground-truth occupation codes are available.")
else:
    raise ValueError(
        "Unavailable ground-truth occupation codes found:\n"
        + unavailable_df.to_string(index=False)
    )

In [ ]:
ground_truth_parts = []

for profile_name, codes in relevant_occupations.items():
    matched = matrices.lookup.loc[
        matrices.lookup["onet_soc_code"].isin(codes),
        ["onet_soc_code", "occupation_title"],
    ].copy()

    matched.insert(0, "Profile", profile_name)
    ground_truth_parts.append(matched)

ground_truth_df = (
    pd.concat(ground_truth_parts, ignore_index=True)
    .sort_values(["Profile", "occupation_title"])
    .reset_index(drop=True)
)

display(ground_truth_df)

print(
    ground_truth_df.groupby("Profile")
    .size()
    .rename("Ground-Truth Occupations")
)

## 8. Run all seven recommenders for all six profiles

Baseline, Weighted Compatibility, and Cosine Similarity each run once per profile. KNN runs once per distance metric (euclidean, manhattan, minkowski, cosine), so each profile produces seven result sets in total.

In [ ]:
model_outputs = {}

knn_metric_variants = {
    "KNN (Euclidean)": {"metric": "euclidean"},
    "KNN (Manhattan)": {"metric": "manhattan"},
    "KNN (Minkowski)": {"metric": "minkowski", "p": 3},
    "KNN (Cosine)": {"metric": "cosine"},
}

for profile in evaluation_profiles:
    profile_name = profile.profile_name

    model_outputs[(profile_name, "Baseline")] = (
        baseline_model.recommend_baseline(
            df=df,
            profile=profile,
            top_n=10,
        )
    )

    model_outputs[(profile_name, "Weighted Compatibility")] = (
        compatibility.compute_weighted_compatibility(
            matrices=matrices,
            scaled=scaled,
            profile=profile,
            top_n=10,
        )
    )

    model_outputs[(profile_name, "Cosine Similarity")] = (
        cosine_model.recommend_cosine(
            matrices=matrices,
            scaled=scaled,
            profile=profile,
            top_n=10,
        )
    )

    for knn_label, knn_kwargs in knn_metric_variants.items():
        model_outputs[(profile_name, knn_label)] = (
            knn_model.recommend_knn(
                matrices=matrices,
                scaled=scaled,
                profile=profile,
                k=10,
                **knn_kwargs,
            )
        )

n_models = 3 + len(knn_metric_variants)
print(
    f"Generated {len(model_outputs)} result sets "
    f"({len(evaluation_profiles)} profiles × {n_models} models)."
)

## 9. Standardize and verify the recommendation results

In [ ]:
standardized_results = []

for (profile_name, model_name), results in model_outputs.items():
    temp = results.copy()

    if "rank" not in temp.columns:
        temp["rank"] = range(1, len(temp) + 1)

    temp["Profile"] = profile_name
    temp["Model"] = model_name

    standardized_results.append(
        temp[
            [
                "Profile",
                "Model",
                "rank",
                "onet_soc_code",
                "occupation_title",
            ]
        ]
    )

recommendations_df = pd.concat(
    standardized_results,
    ignore_index=True,
)

print("Combined table shape:", recommendations_df.shape)
display(recommendations_df.head(20))

In [ ]:
counts_check = (
    recommendations_df
    .groupby(["Profile", "Model"])
    .size()
    .reset_index(name="Recommendation Count")
)

display(counts_check)

all_counts_valid = counts_check["Recommendation Count"].eq(10).all()
print("All combinations contain 10 recommendations:", all_counts_valid)

if not all_counts_valid:
    raise ValueError("At least one profile-model combination does not contain 10 results.")

## 10. Define the evaluation metrics

The evaluation includes:

- Precision@5 and Precision@10
- Recall@5 and Recall@10
- Hit Rate@5 and Hit Rate@10
- Mean Reciprocal Rank (MRR)
- Normalized Discounted Cumulative Gain at 10 (NDCG@10)
- Catalog Coverage

In [ ]:
def precision_at_k(recommended_codes, relevant_codes, k):
    top_k = recommended_codes[:k]
    return len(set(top_k) & relevant_codes) / k


def recall_at_k(recommended_codes, relevant_codes, k):
    top_k = recommended_codes[:k]
    return len(set(top_k) & relevant_codes) / len(relevant_codes)


def hit_rate_at_k(recommended_codes, relevant_codes, k):
    top_k = recommended_codes[:k]
    return float(any(code in relevant_codes for code in top_k))


def reciprocal_rank(recommended_codes, relevant_codes):
    for rank, code in enumerate(recommended_codes, start=1):
        if code in relevant_codes:
            return 1 / rank
    return 0.0


def ndcg_at_k(recommended_codes, relevant_codes, k):
    top_k = recommended_codes[:k]

    relevance = [
        1 if code in relevant_codes else 0
        for code in top_k
    ]

    dcg = sum(
        rel / np.log2(rank + 1)
        for rank, rel in enumerate(relevance, start=1)
    )

    ideal_hits = min(len(relevant_codes), k)
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg if idcg > 0 else 0.0

## 11. Calculate profile-level recommendation metrics

In [ ]:
metric_rows = []

for (profile_name, model_name), group in recommendations_df.groupby(
    ["Profile", "Model"]
):
    group = group.sort_values("rank")
    recommended_codes = group["onet_soc_code"].tolist()
    relevant_codes = relevant_occupations[profile_name]

    metric_rows.append(
        {
            "Profile": profile_name,
            "Model": model_name,
            "Precision@5": precision_at_k(
                recommended_codes, relevant_codes, 5
            ),
            "Precision@10": precision_at_k(
                recommended_codes, relevant_codes, 10
            ),
            "Recall@5": recall_at_k(
                recommended_codes, relevant_codes, 5
            ),
            "Recall@10": recall_at_k(
                recommended_codes, relevant_codes, 10
            ),
            "Hit Rate@5": hit_rate_at_k(
                recommended_codes, relevant_codes, 5
            ),
            "Hit Rate@10": hit_rate_at_k(
                recommended_codes, relevant_codes, 10
            ),
            "MRR": reciprocal_rank(
                recommended_codes, relevant_codes
            ),
            "NDCG@10": ndcg_at_k(
                recommended_codes, relevant_codes, 10
            ),
        }
    )

profile_metrics_df = pd.DataFrame(metric_rows)

display(
    profile_metrics_df.sort_values(
        ["Profile", "Model"]
    ).reset_index(drop=True)
)

## 12. Calculate catalog coverage and recommendation diversity

In [ ]:
catalog_size = len(matrices.lookup)

coverage_rows = []

for model_name, group in recommendations_df.groupby("Model"):
    unique_recommended = group["onet_soc_code"].nunique()

    coverage_rows.append(
        {
            "Model": model_name,
            "Unique Occupations Recommended": unique_recommended,
            "Catalog Size": catalog_size,
            "Catalog Coverage": unique_recommended / catalog_size,
        }
    )

coverage_df = pd.DataFrame(coverage_rows)

display(coverage_df.sort_values("Catalog Coverage", ascending=False))

## 13. Create the final model-comparison table

In [ ]:
model_comparison_df = (
    profile_metrics_df
    .groupby("Model", as_index=False)
    .agg(
        {
            "Precision@5": "mean",
            "Precision@10": "mean",
            "Recall@5": "mean",
            "Recall@10": "mean",
            "Hit Rate@5": "mean",
            "Hit Rate@10": "mean",
            "MRR": "mean",
            "NDCG@10": "mean",
        }
    )
    .merge(
        coverage_df[["Model", "Catalog Coverage"]],
        on="Model",
        how="left",
    )
)

metric_columns = [
    "Precision@5",
    "Precision@10",
    "Recall@5",
    "Recall@10",
    "Hit Rate@5",
    "Hit Rate@10",
    "MRR",
    "NDCG@10",
    "Catalog Coverage",
]

model_comparison_df[metric_columns] = (
    model_comparison_df[metric_columns].round(3)
)

display(model_comparison_df)

## 14. Stability and diversity summary

In [ ]:
diversity_summary_df = (
    recommendations_df
    .groupby("Model")["occupation_title"]
    .nunique()
    .rename("Unique Occupations Recommended")
    .reset_index()
    .merge(
        coverage_df[["Model", "Catalog Coverage"]],
        on="Model",
        how="left",
    )
    .sort_values("Unique Occupations Recommended", ascending=False)
    .reset_index(drop=True)
)

diversity_summary_df["Catalog Coverage"] = (
    diversity_summary_df["Catalog Coverage"].round(3)
)

display(diversity_summary_df)

## 15. Optional: export final evaluation tables

In [ ]:
recommendations_df.to_csv(
    "person3_all_recommendations.csv",
    index=False,
)

profile_metrics_df.to_csv(
    "person3_profile_metrics.csv",
    index=False,
)

model_comparison_df.to_csv(
    "person3_model_comparison.csv",
    index=False,
)

diversity_summary_df.to_csv(
    "person3_diversity_summary.csv",
    index=False,
)

print("Evaluation tables exported successfully.")